In [1]:
import os

In [9]:
%pwd

'd:\\Cdac_project\\Wine_prediction_e2e'

In [8]:
os.chdir("../")

In [15]:
# creating entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [16]:
#configuration manager in src config

from Wine_prediction_e2e.constants import *
from Wine_prediction_e2e.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = Config_yaml_path,
        params_filepath = params_yaml_path,
        schema_filepath = schema_yaml_path):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path = config.train_data_path,
            test_data_path = config.test_data_path,
            model_name = config.model_name,
            alpha = params.alpha,
            l1_ratio = params.l1_ratio,
            target_column = schema.name
            
        )

        return model_trainer_config

In [17]:
import pandas as pd
import os
from Wine_prediction_e2e import logger
from sklearn.linear_model import ElasticNet
import joblib

In [18]:
# creating components here

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)


        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y = test_data[[self.config.target_column]]


        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)

        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))

In [19]:
# pipeline

try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2025-11-01 17:36:58,094: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-11-01 17:36:58,097: INFO: common: yaml file: params.yaml loaded successfully]
[2025-11-01 17:36:58,099: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-11-01 17:36:58,101: INFO: common: created directory at: artifacts]
[2025-11-01 17:36:58,101: INFO: common: created directory at: artifacts/model_trainer]


In [7]:
%pwd

'd:\\Cdac_project\\Wine_prediction_e2e\\research'